In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

X_list = []
y_list = []

chunk_size = 100000

chunks = pd.read_csv('US_Accidents_March23.csv', chunksize=chunk_size)

for chunk in chunks:
    condition_counts = chunk['Weather_Condition'].value_counts()
    filtered_chunk = chunk[chunk['Weather_Condition'].isin(condition_counts[condition_counts > 100].index)]

    filtered_chunk = filtered_chunk.dropna(subset=['Temperature(F)', 'Wind_Speed(mph)', 'Severity', 'Start_Lat', 'Start_Lng'])

    filtered_chunk['Start_Time'] = pd.to_datetime(filtered_chunk['Start_Time'], errors='coerce')
    filtered_chunk['End_Time'] = pd.to_datetime(filtered_chunk['End_Time'], errors='coerce')
    filtered_chunk['Start_Hour'] = filtered_chunk['Start_Time'].dt.hour
    filtered_chunk['End_Hour'] = filtered_chunk['End_Time'].dt.hour

    categorical_selected = ['Description', 'City', 'Weather_Condition']
    numeric_selected = ['Distance(mi)', 'Wind_Speed(mph)', 'Temperature(F)', 'Start_Lat', 'Start_Lng', 'Start_Hour', 'End_Hour']
    selected_features = categorical_selected + numeric_selected

    X_chunk = filtered_chunk[selected_features]
    y_chunk = filtered_chunk['Severity']

    X_list.append(X_chunk)
    y_list.append(y_chunk)

X = pd.concat(X_list, axis=0)
y = pd.concat(y_list, axis=0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()
le = LabelEncoder()

for feature in categorical_selected:
    if feature in X_train.columns:
        le.fit(pd.concat([X_train[feature], X_test[feature]]).astype(str))
        X_train_encoded[feature] = le.transform(X_train[feature].astype(str))
        X_test_encoded[feature] = le.transform(X_test[feature].astype(str))

scaler = StandardScaler()
X_train_encoded[numeric_selected] = scaler.fit_transform(X_train_encoded[numeric_selected])
X_test_encoded[numeric_selected] = scaler.transform(X_test_encoded[numeric_selected])

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_encoded, y_train)

y_pred = rf_model.predict(X_test_encoded)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9398623411070175
Classification Report:
               precision    recall  f1-score   support

           1       0.87      0.65      0.75     13136
           2       0.96      0.97      0.96   1137257
           3       0.88      0.85      0.87    226815
           4       0.77      0.55      0.64     35850

    accuracy                           0.94   1413058
   macro avg       0.87      0.76      0.80   1413058
weighted avg       0.94      0.94      0.94   1413058



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from datetime import datetime

samples_per_class = 3000
chunk_size = 100000
sampled_data = {1: [], 2: [], 3: [], 4: []}

print(" Sampling balanced data...")
for chunk in pd.read_csv('US_Accidents_March23.csv', chunksize=chunk_size):
    chunk = chunk.dropna(subset=['Severity', 'Temperature(F)', 'Wind_Speed(mph)', 'Weather_Condition', 'City', 'Start_Time', 'End_Time'])

    for severity in [1, 2, 3, 4]:
        needed = samples_per_class - len(sampled_data[severity])
        if needed > 0:
            subset = chunk[chunk['Severity'] == severity]
            if not subset.empty:
                sampled = subset.sample(n=min(needed, len(subset)), random_state=42)
                sampled_data[severity].append(sampled)

    if all(len(sampled_data[s]) >= samples_per_class for s in [1, 2, 3, 4]):
        break

df = pd.concat([pd.concat(sampled_data[s]) for s in [1, 2, 3, 4]], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(" Balanced dataset prepared.")

df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')
df['End_Time'] = pd.to_datetime(df['End_Time'], errors='coerce')

df['Start_Hour'] = df['Start_Time'].dt.hour
df['End_Hour'] = df['End_Time'].dt.hour

categorical_selected = ['City', 'Weather_Condition']
numeric_selected = ['Wind_Speed(mph)', 'Temperature(F)', 'Start_Hour', 'End_Hour']
selected_features = categorical_selected + numeric_selected

df = df[selected_features + ['Severity']].dropna()

X = df[selected_features]
y = df['Severity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

le_dict = {}
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

for feature in categorical_selected:
    le = LabelEncoder()
    le.fit(pd.concat([X_train[feature], X_test[feature]]).astype(str))
    X_train_encoded[feature] = le.transform(X_train[feature].astype(str))
    X_test_encoded[feature] = le.transform(X_test[feature].astype(str))
    le_dict[feature] = le
    
scaler = StandardScaler()
X_train_encoded[numeric_selected] = scaler.fit_transform(X_train_encoded[numeric_selected])
X_test_encoded[numeric_selected] = scaler.transform(X_test_encoded[numeric_selected])

model = RandomForestClassifier(random_state=42)
model.fit(X_train_encoded, y_train)

y_pred = model.predict(X_test_encoded)
print(" Accuracy:", accuracy_score(y_test, y_pred))
print(" Classification Report:\n", classification_report(y_test, y_pred))

print("\n Enter details to predict accident severity:")

input_city = input("City: ")
input_weather = input("Weather Condition: ")
input_temp = float(input("Temperature (F): "))
input_wind = float(input("Wind Speed (mph): "))
input_start_time = input("Start Time (HH:MM in 24hr format): ")
input_end_time = input("End Time (HH:MM in 24hr format): ")

try:
    input_start_hour = datetime.strptime(input_start_time, "%H:%M").hour
except ValueError:
    print(" Invalid start time format. Using hour = 12.")
    input_start_hour = 12

try:
    input_end_hour = datetime.strptime(input_end_time, "%H:%M").hour
except ValueError:
    print(" Invalid end time format. Using hour = 13.")
    input_end_hour = 13

input_df = pd.DataFrame([{
    'City': input_city,
    'Weather_Condition': input_weather,
    'Wind_Speed(mph)': input_wind,
    'Temperature(F)': input_temp,
    'Start_Hour': input_start_hour,
    'End_Hour': input_end_hour
}])

for feature in categorical_selected:
    if input_df[feature].iloc[0] in le_dict[feature].classes_:
        input_df[feature] = le_dict[feature].transform(input_df[feature].astype(str))
    else:
        print(f" Warning: '{input_df[feature].iloc[0]}' not seen during training for '{feature}'. Using default value.")
        input_df[feature] = le_dict[feature].transform([le_dict[feature].classes_[0]])

input_df[numeric_selected] = scaler.transform(input_df[numeric_selected])

predicted_severity = model.predict(input_df)[0]
print(f" Predicted Severity: {predicted_severity}")

 Sampling balanced data...
 Balanced dataset prepared.
 Accuracy: 0.5254100364594959
 Classification Report:
               precision    recall  f1-score   support

           1       0.43      0.19      0.26      3248
           2       0.54      0.64      0.58     41638
           3       0.48      0.43      0.46     27373
           4       0.56      0.48      0.52     21818

    accuracy                           0.53     94077
   macro avg       0.50      0.43      0.45     94077
weighted avg       0.52      0.53      0.52     94077


 Enter details to predict accident severity:
